<div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 50%,#0f3460 100%);padding:40px 30px;border-radius:12px;margin-bottom:8px">
  <div style="display:flex;align-items:center;gap:20px;flex-wrap:wrap">
    <div style="background:#e94560;border-radius:50%;width:64px;height:64px;display:flex;align-items:center;justify-content:center;font-size:30px;flex-shrink:0">&#127891;</div>
    <div>
      <h1 style="color:#ffffff;margin:0;font-size:26px;font-weight:800;letter-spacing:1px">AI IMPACT ON STUDENTS</h1>
      <p style="color:#a8b2d8;margin:4px 0 0;font-size:13px">Data Analytics &amp; Machine Learning Project &nbsp;|&nbsp; IBM SkillsBuild</p>
    </div>
  </div>
  <hr style="border:none;border-top:1px solid rgba(255,255,255,0.15);margin:20px 0" />
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;color:#cdd6f4;font-size:13px">
    <div>&#128100; <b>Author:</b> Dharamveer Sharma</div>
    <div>&#127981; <b>Program:</b> IBM SkillsBuild Data Analytics with AI (BharatCares x AICTE)</div>
    <div>&#128193; <b>Repository:</b> IBM_Skillbuild_project</div>
    <div>&#128202; <b>Dataset:</b> AI Impact on Students (Kaggle) &mdash; 50,000 records</div>
    <div style="grid-column:1/-1">&#127919; <b>Goal:</b> Predict student burnout risk and analyse how GenAI usage, study habits, AI dependency, and prompt skills affect academic outcomes</div>
  </div>
</div>

<div style="background:#f0f4ff;border-left:5px solid #0f3460;padding:14px 20px;border-radius:6px;font-size:13px">
<b style="color:#0f3460;font-size:14px">&#128203; TABLE OF CONTENTS</b>
<ol style="margin:8px 0 0;padding-left:22px;color:#333;line-height:2.1">
<li>Environment Setup &mdash; install all dependencies</li>
<li>Import Libraries</li>
<li>Data Loading &amp; Preview</li>
<li>Exploratory Overview (shape, dtypes, missing values, class balance)</li>
<li>Data Cleaning</li>
<li>Feature Engineering (5 new features)</li>
<li>Exploratory Data Analysis &mdash; 5 static visualisations + 1 interactive Plotly chart</li>
<li>Machine Learning &mdash; Random Forest + Logistic Regression, confusion matrices, feature importance</li>
<li>Key Analytical Insights (styled summary card)</li>
<li>Conclusion &amp; Ethical Recommendations</li>
</ol>
</div>

<div style="background:#fff3cd;border:1px solid #ffc107;padding:10px 16px;border-radius:6px;font-size:13px;margin-top:12px">
&#9888;&#65039; <b>STEP 0 &mdash; Environment Setup</b><br>
Run this cell <b>once</b> the very first time. It installs all required packages into your Python environment.
Once installed, you can skip it on future runs.
</div>

In [ ]:
# ============================================================
# STEP 0 — Install all project dependencies
# Run once; safe to re-run (pip skips already-installed pkgs)
# ============================================================
import subprocess, sys

PACKAGES = [
    'pandas>=2.0',
    'numpy>=1.24',
    'scikit-learn>=1.3',
    'matplotlib>=3.7',
    'seaborn>=0.12',
    'plotly>=5.18',
    'ipywidgets>=8.0',
]

print('Installing / verifying dependencies...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + PACKAGES,
    capture_output=True, text=True
)
if result.returncode == 0:
    print('All packages ready.')
else:
    print('pip output:', result.stderr[-800:])

## Step 1 — Import Libraries

In [ ]:
# ── Core ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Machine Learning ───────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score
)

# ── Display / IPython ──────────────────────────────────────────────────────
from IPython.display import display, HTML

# ── Reproducibility ────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Notebook aesthetics ────────────────────────────────────────────────────
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

print('Libraries loaded successfully.')
print(f'  pandas {pd.__version__}  |  numpy {np.__version__}  |  sklearn available')

## Step 2 — Data Loading

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────
# The CSV must be in the same directory as this notebook.
# Source: Kaggle — AI Impact on Students Dataset
# ──────────────────────────────────────────────────────────────────────────
import os

CSV_PATH = 'ai_student_impact_dataset (1).csv'

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f'Dataset not found: {CSV_PATH}\n'
        'Place the CSV in the same folder as this notebook and re-run.'
    )

df_raw = pd.read_csv(CSV_PATH)

print(f'Dataset loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

## Step 3 — Exploratory Overview

In [ ]:
df_raw.info()

In [ ]:
# Statistical summary of all numeric features
df_raw.describe().round(3)

In [ ]:
# ── Missing values check ───────────────────────────────────────────────────
missing = df_raw.isnull().sum()
if missing.sum() == 0:
    print('No missing values found in any column.')
else:
    print('Missing values per column:')
    print(missing[missing > 0])

In [ ]:
# ── Target class distribution ──────────────────────────────────────────────
counts  = df_raw['Burnout_Risk_Level'].value_counts()
props   = df_raw['Burnout_Risk_Level'].value_counts(normalize=True).mul(100).round(2)

display(HTML(
    '<div style="background:#f8f9fa;border-radius:8px;padding:14px 20px;font-family:monospace;font-size:13px">'
    '<b>Target Variable: Burnout_Risk_Level</b><br><br>'
    + ''.join(
        f'<span style="display:inline-block;width:140px">{lvl}</span>'
        f'<span style="display:inline-block;width:80px;text-align:right">{counts[lvl]:,}</span>'
        f'<span style="display:inline-block;margin-left:16px">{props[lvl]}%</span>'
        f'  <span style="color:{'#2ecc71' if lvl=='Low' else '#f39c12' if lvl=='Medium' else '#e74c3c'}">{chr(9646)*int(props[lvl]//2)}</span><br>'
        for lvl in ['Low','Medium','High']
    )
    + '</div>'
))
print(f'Majority-class baseline accuracy: {props.max()/100:.4f}')

## Step 4 — Data Cleaning

In [ ]:
# Work on a copy so the raw frame is always preserved
df = df_raw.copy()

# ── 4.1  Drop identifier column (no analytical value) ──────────────────────
df.drop(columns=['Student_ID'], errors='ignore', inplace=True)
print('4.1 Student_ID dropped.')

# ── 4.2  Validate GPA columns are within [0, 4.0] ─────────────────────────
for col in ['Pre_Semester_GPA', 'Post_Semester_GPA']:
    out_of_range = df[(df[col] < 0) | (df[col] > 4.0)]
    print(f'4.2 {col} — rows outside [0,4]: {len(out_of_range)}')

# ── 4.3  Ensure boolean type for Paid_Subscription ────────────────────────
df['Paid_Subscription'] = df['Paid_Subscription'].astype(bool)

# ── 4.4  Strip whitespace from all categorical columns ────────────────────
cat_cols = [
    'Major_Category', 'Year_of_Study', 'Primary_Use_Case',
    'Prompt_Engineering_Skill', 'Institutional_Policy', 'Burnout_Risk_Level'
]
for col in cat_cols:
    df[col] = df[col].str.strip()

print(f'\nCleaning complete. Final shape: {df.shape}')
df.head(3)

## Step 5 — Feature Engineering

In [ ]:
# ── 5.1  GPA Change — academic trajectory over the semester ───────────────
df['GPA_Change'] = (df['Post_Semester_GPA'] - df['Pre_Semester_GPA']).round(3)

# ── 5.2  AI-to-Study Ratio — fraction of total study time on GenAI ────────
#         Adding epsilon avoids division-by-zero when both hours are 0
df['AI_Study_Ratio'] = (
    df['Weekly_GenAI_Hours'] /
    (df['Traditional_Study_Hours'] + df['Weekly_GenAI_Hours'] + 1e-6)
).round(4)

# ── 5.3  Skill Efficiency — retained learning per AI hour ─────────────────
df['Skill_Efficiency'] = (
    df['Skill_Retention_Score'] / (df['Weekly_GenAI_Hours'] + 1)
).round(4)

# ── 5.4  High AI Dependency flag (binary: dependency >= 5) ────────────────
df['High_AI_Dependency'] = (df['Perceived_AI_Dependency'] >= 5).astype(int)

# ── 5.5  Prompt Skill numeric (for correlation analysis) ──────────────────
df['Prompt_Skill_Numeric'] = df['Prompt_Engineering_Skill'].map(
    {'Beginner': 1, 'Intermediate': 2, 'Advanced': 3}
)

new_features = ['GPA_Change', 'AI_Study_Ratio', 'Skill_Efficiency',
                'High_AI_Dependency', 'Prompt_Skill_Numeric']
print('Engineered features summary:')
df[new_features].describe().round(3)

<div style="display:flex;align-items:center;gap:0;margin:24px 0 8px;font-size:12px;font-weight:700;font-family:sans-serif">
  <div style="background:#0f3460;color:#fff;padding:8px 18px;border-radius:6px 0 0 6px">&#10003; Steps 1&#8211;5 Complete</div>
  <div style="background:#e94560;color:#fff;padding:8px 18px">&#9654; Step 6: EDA</div>
  <div style="background:#dee2e6;color:#555;padding:8px 18px">Step 7: ML</div>
  <div style="background:#dee2e6;color:#555;padding:8px 18px;border-radius:0 6px 6px 0">Steps 8&#8211;9: Insights</div>
</div>

## Step 6 — Exploratory Data Analysis

### 6.1 GPA Change Distribution

In [ ]:
order   = ['Low', 'Medium', 'High']
palette = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram + KDE
sns.histplot(df['GPA_Change'], bins=40, kde=True, color='steelblue', ax=axes[0])
axes[0].axvline(0, color='red', linestyle='--', lw=1.3, label='Zero change')
axes[0].set_title('Distribution of GPA Change', fontweight='bold')
axes[0].set_xlabel('GPA Change (Post − Pre)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot stratified by burnout risk
sns.boxplot(data=df, x='Burnout_Risk_Level', y='GPA_Change',
            order=order, palette=palette, ax=axes[1])
axes[1].axhline(0, color='grey', linestyle='--', lw=0.9)
axes[1].set_title('GPA Change by Burnout Risk Level', fontweight='bold')
axes[1].set_xlabel('Burnout Risk Level')
axes[1].set_ylabel('GPA Change')

plt.suptitle('GPA Change Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_gpa_change.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_gpa_change.png')

### 6.2 AI Dependency vs Skill Retention

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: dependency vs retention, coloured by burnout risk
bp = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
for level in ['Low', 'Medium', 'High']:
    sub = df[df['Burnout_Risk_Level'] == level]
    axes[0].scatter(
        sub['Perceived_AI_Dependency'], sub['Skill_Retention_Score'],
        label=level, alpha=0.35, s=8, color=bp[level]
    )
z = np.polyfit(df['Perceived_AI_Dependency'], df['Skill_Retention_Score'], 1)
xl = np.linspace(df['Perceived_AI_Dependency'].min(), df['Perceived_AI_Dependency'].max(), 100)
axes[0].plot(xl, np.poly1d(z)(xl), 'k--', lw=1.5, label='Trend')
axes[0].set_title('AI Dependency vs Skill Retention', fontweight='bold')
axes[0].set_xlabel('Perceived AI Dependency (1–10)')
axes[0].set_ylabel('Skill Retention Score')
axes[0].legend(title='Burnout Risk', markerscale=3)

# Bar: mean skill retention by prompt engineering level
avg = df.groupby('Prompt_Engineering_Skill')['Skill_Retention_Score'].mean()
avg = avg.reindex(['Beginner', 'Intermediate', 'Advanced'])
colors_bar = ['#e74c3c', '#f39c12', '#2ecc71']
bars = axes[1].bar(avg.index, avg.values, color=colors_bar, edgecolor='black', alpha=0.85)
for bar, val in zip(bars, avg.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=10)
axes[1].set_title('Avg Skill Retention by Prompt Skill Level', fontweight='bold')
axes[1].set_xlabel('Prompt Engineering Skill')
axes[1].set_ylabel('Average Skill Retention Score')
axes[1].set_ylim(0, 100)

plt.suptitle('AI Dependency & Prompt Skills', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_ai_dependency_skill.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_ai_dependency_skill.png')

### 6.3 Weekly GenAI Hours vs GPA Change

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Regression scatter
sns.regplot(data=df, x='Weekly_GenAI_Hours', y='GPA_Change',
            scatter_kws={'alpha': 0.2, 's': 8, 'color': 'steelblue'},
            line_kws={'color': 'red', 'lw': 1.8}, ax=axes[0])
axes[0].axhline(0, color='grey', linestyle='--', lw=0.9)
axes[0].set_title('Weekly GenAI Hours vs GPA Change', fontweight='bold')
axes[0].set_xlabel('Weekly GenAI Hours')
axes[0].set_ylabel('GPA Change')

# Stacked bar by major
pivot = (
    df.groupby(['Major_Category', 'Burnout_Risk_Level']).size()
      .unstack(fill_value=0)
      .reindex(columns=['Low', 'Medium', 'High'])
)
pivot.plot(kind='bar', ax=axes[1],
           color=['#2ecc71', '#f39c12', '#e74c3c'],
           edgecolor='black', alpha=0.85)
axes[1].set_title('Burnout Risk Count by Major', fontweight='bold')
axes[1].set_xlabel('Major Category')
axes[1].set_ylabel('Number of Students')
axes[1].legend(title='Burnout Risk')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('GenAI Usage, GPA & Burnout by Major', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_genai_gpa_major.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_genai_gpa_major.png')

### 6.4 Correlation Heatmap

In [ ]:
num_cols = [
    'Pre_Semester_GPA', 'Post_Semester_GPA', 'Weekly_GenAI_Hours',
    'Traditional_Study_Hours', 'Perceived_AI_Dependency',
    'Anxiety_Level_During_Exams', 'Skill_Retention_Score', 'Tool_Diversity',
    'GPA_Change', 'AI_Study_Ratio', 'Skill_Efficiency', 'Prompt_Skill_Numeric'
]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))   # lower triangle only
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_correlation_heatmap.png')

### 6.5 Institutional Policy Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
policy_order = ['Strict_Ban', 'Allowed_With_Citation', 'Actively_Encouraged']

# Violin: AI-to-Study ratio by policy
sns.violinplot(data=df, x='Institutional_Policy', y='AI_Study_Ratio',
               order=policy_order, palette='Set2', ax=axes[0])
axes[0].set_title('AI-to-Study Ratio by Policy', fontweight='bold')
axes[0].set_xlabel('Institutional Policy')
axes[0].set_ylabel('AI Study Ratio')
axes[0].tick_params(axis='x', rotation=15)

# Bar: avg exam anxiety by policy x burnout
ap = (
    df.groupby(['Institutional_Policy', 'Burnout_Risk_Level'])['Anxiety_Level_During_Exams']
      .mean().unstack().reindex(columns=['Low', 'Medium', 'High'])
)
ap.plot(kind='bar', ax=axes[1],
        color=['#2ecc71', '#f39c12', '#e74c3c'],
        edgecolor='black', alpha=0.85)
axes[1].set_title('Avg Exam Anxiety: Policy x Burnout', fontweight='bold')
axes[1].set_xlabel('Institutional Policy')
axes[1].set_ylabel('Average Anxiety Level')
axes[1].legend(title='Burnout Risk')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Institutional Policy Impact', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_policy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_policy_analysis.png')

### 6.6 Interactive Chart — Burnout by GenAI Hours (Plotly)

In [ ]:
# Plotly scatter — hover over any dot for full student profile
fig_px = px.scatter(
    df.sample(3000, random_state=42),   # sample for performance
    x='Weekly_GenAI_Hours',
    y='GPA_Change',
    color='Burnout_Risk_Level',
    symbol='Prompt_Engineering_Skill',
    size='Skill_Retention_Score',
    size_max=12,
    hover_data=['Major_Category', 'Perceived_AI_Dependency',
                'Traditional_Study_Hours', 'Anxiety_Level_During_Exams'],
    color_discrete_map={'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'},
    title='GenAI Hours vs GPA Change (hover for details)',
    labels={
        'Weekly_GenAI_Hours': 'Weekly GenAI Hours',
        'GPA_Change': 'GPA Change (Post - Pre)',
        'Burnout_Risk_Level': 'Burnout Risk',
        'Prompt_Engineering_Skill': 'Prompt Skill'
    },
    template='plotly_white',
    height=520
)
fig_px.add_hline(y=0, line_dash='dash', line_color='grey',
                 annotation_text='No GPA change', annotation_position='right')
fig_px.update_layout(
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    title_font_size=15
)
fig_px.show()
print('Tip: Hover over any point to see the full student profile.')

### 6.7 Interactive Dataset Explorer (ipywidgets)


In [ ]:
# ── Interactive Dataset Explorer ──────────────────────────────────────────
# Filter students by Major Category and Year of Study.
# The KDE + boxplot update instantly when you change any control.
# Requires: ipywidgets (installed via Step 0)
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import matplotlib.pyplot as plt, seaborn as sns

    majors  = ['All'] + sorted(df['Major_Category'].unique().tolist())
    years   = ['All', 'Freshman', 'Sophomore', 'Junior', 'Senior', 'Graduate']
    metrics = ['GPA_Change', 'Skill_Retention_Score', 'Weekly_GenAI_Hours', 'AI_Study_Ratio']

    w_major  = widgets.Dropdown(options=majors,  value='All',        description='Major:',
                                style={'description_width':'70px'}, layout=widgets.Layout(width='220px'))
    w_year   = widgets.Dropdown(options=years,   value='All',        description='Year:',
                                style={'description_width':'70px'}, layout=widgets.Layout(width='220px'))
    w_metric = widgets.Dropdown(options=metrics, value='GPA_Change', description='Metric:',
                                style={'description_width':'70px'}, layout=widgets.Layout(width='220px'))
    out = widgets.Output()

    def update_chart(change=None):
        with out:
            clear_output(wait=True)
            sub = df.copy()
            if w_major.value != 'All':
                sub = sub[sub['Major_Category'] == w_major.value]
            if w_year.value != 'All':
                sub = sub[sub['Year_of_Study'] == w_year.value]
            metric  = w_metric.value
            palette = {'Low':'#2ecc71','Medium':'#f39c12','High':'#e74c3c'}
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            for lvl, col in palette.items():
                g = sub[sub['Burnout_Risk_Level'] == lvl][metric]
                if len(g) > 5:
                    sns.kdeplot(g, ax=axes[0], label=lvl, color=col, fill=True, alpha=0.3)
            axes[0].set_title(f'{metric} — KDE by Burnout Risk', fontweight='bold')
            axes[0].set_xlabel(metric)
            axes[0].legend(title='Burnout Risk')
            sns.boxplot(data=sub, x='Burnout_Risk_Level', y=metric,
                        order=['Low','Medium','High'], palette=palette, ax=axes[1])
            axes[1].set_title(f'{metric} — Box Plot', fontweight='bold')
            fig.suptitle(f'Major={w_major.value}  |  Year={w_year.value}  |  n={len(sub):,} students',
                         fontsize=11, fontweight='bold', y=1.02)
            plt.tight_layout()
            plt.show()

    for w in [w_major, w_year, w_metric]:
        w.observe(update_chart, names='value')

    display(widgets.HBox(
        [widgets.VBox([w_major, w_year, w_metric],
                      layout=widgets.Layout(border='1px solid #dee2e6', padding='10px',
                                            border_radius='8px', background='#f8f9fa',
                                            width='250px'))],
    ))
    display(out)
    update_chart()

except ImportError:
    print('ipywidgets not installed. Run Step 0 first, then restart the kernel.')


<div style="display:flex;align-items:center;gap:0;margin:24px 0 8px;font-size:12px;font-weight:700;font-family:sans-serif">
  <div style="background:#0f3460;color:#fff;padding:8px 18px;border-radius:6px 0 0 6px">&#10003; Steps 1&#8211;6 Complete</div>
  <div style="background:#e94560;color:#fff;padding:8px 18px">&#9654; Step 7: Machine Learning</div>
  <div style="background:#dee2e6;color:#555;padding:8px 18px;border-radius:0 6px 6px 0">Steps 8&#8211;9: Insights</div>
</div>

## Step 7 — Machine Learning: Burnout Risk Prediction

### 7.1 Feature Encoding

In [ ]:
ml_df = df.copy()

# Ordinal encoding (order is meaningful)
ordinal_maps = {
    'Year_of_Study':           {'Freshman':1,'Sophomore':2,'Junior':3,'Senior':4,'Graduate':5},
    'Prompt_Engineering_Skill':{'Beginner':1,'Intermediate':2,'Advanced':3},
    'Burnout_Risk_Level':      {'Low':0,'Medium':1,'High':2},
}
for col, mapping in ordinal_maps.items():
    ml_df[col] = ml_df[col].map(mapping)

# Boolean → integer
ml_df['Paid_Subscription'] = ml_df['Paid_Subscription'].astype(int)

# One-Hot Encoding for nominal features
nominal_cols = ['Major_Category', 'Primary_Use_Case', 'Institutional_Policy']
ml_df = pd.get_dummies(ml_df, columns=nominal_cols, drop_first=True)

# Verify no NaN values remain after encoding
nan_count = ml_df.isnull().sum().sum()
print(f'Encoded shape : {ml_df.shape}')
print(f'NaN after encoding: {nan_count}  (must be 0)')
print(f'Target class counts:')
print(ml_df['Burnout_Risk_Level'].value_counts().sort_index())

### 7.2 Train / Test Split

In [ ]:
TARGET   = 'Burnout_Risk_Level'
FEATURES = [c for c in ml_df.columns if c != TARGET]

X = ml_df[FEATURES].astype(float)   # ensure float for scaler compatibility
y = ml_df[TARGET]

# Stratified 80/20 split preserves class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Testing  samples : {X_test.shape[0]:,}')
print(f'Feature count    : {X.shape[1]}')

### 7.3 Model 1 — Random Forest Classifier

In [14]:
# Random Forest handles mixed-type tabular data and non-linear relationships.
# Note: On this intentionally-noisy synthetic dataset (~50k rows),
# a meaningful accuracy above the 42% majority-class baseline confirms
# the model has learned real signal.

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,          # unlimited depth — best bias/variance for this data
    min_samples_leaf=2,
    class_weight='balanced', # compensates for class imbalance
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
rf_acc    = accuracy_score(y_test, y_pred_rf)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rf_cv = cross_val_score(rf_model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

label_names = ['Low', 'Medium', 'High']

print(f'Random Forest — Test Accuracy : {rf_acc:.4f}')
print(f'Random Forest — CV Accuracy   : {rf_cv.mean():.4f} +/- {rf_cv.std():.4f}')
print(f'Majority-class baseline       : 0.4229')
print(f'Improvement over baseline     : +{(rf_acc - 0.4229)*100:.1f} pp\n')
print('Classification Report (Random Forest):')
print(classification_report(y_test, y_pred_rf, target_names=label_names))

NameError: name 'X_train' is not defined

### 7.4 Model 2 — Logistic Regression (Baseline)

In [ ]:
# Logistic Regression: interpretable linear baseline.
# Note: 'multi_class' parameter was removed in scikit-learn 1.5+;
# multinomial logistic regression is used automatically for 3+ classes.

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr_model = LogisticRegression(
    max_iter=1000,           # increased from 500 — ensures convergence
    class_weight='balanced',
    random_state=RANDOM_STATE,
    solver='lbfgs'           # efficient for multiclass problems
    # multi_class removed — deprecated/removed in sklearn >= 1.5
)
lr_model.fit(X_train_s, y_train)

y_pred_lr = lr_model.predict(X_test_s)
lr_acc    = accuracy_score(y_test, y_pred_lr)

lr_cv = cross_val_score(
    lr_model, scaler.fit_transform(X), y,
    cv=cv, scoring='accuracy', n_jobs=-1
)

print(f'Logistic Regression — Test Accuracy : {lr_acc:.4f}')
print(f'Logistic Regression — CV Accuracy   : {lr_cv.mean():.4f} +/- {lr_cv.std():.4f}\n')
print('Classification Report (Logistic Regression):')
print(classification_report(y_test, y_pred_lr, target_names=label_names))

### 7.5 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_rf, y_pred_lr],
    ['Random Forest', 'Logistic Regression']
):
    cm   = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Confusion Matrix — {title}', fontweight='bold', fontsize=12)

plt.suptitle('Model Evaluation: Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_confusion_matrices.png')

### 7.6 Feature Importance

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=FEATURES)
top20       = importances.sort_values(ascending=False).head(20)

colors = sns.color_palette('RdYlGn', len(top20))[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
top20[::-1].plot(kind='barh', ax=ax, color=colors, edgecolor='black', alpha=0.85)
ax.set_title('Top 20 Feature Importances — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('plot_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_feature_importance.png')
print('\nTop 10 most important features:')
print(top20.head(10).to_string())

### 7.7 Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model':              ['Random Forest', 'Logistic Regression', 'Majority-class Baseline'],
    'Test Accuracy':      [round(rf_acc,4),  round(lr_acc,4),       0.4229],
    'CV Mean Accuracy':   [round(rf_cv.mean(),4), round(lr_cv.mean(),4), 0.4229],
    'CV Std Dev':         [round(rf_cv.std(),4),  round(lr_cv.std(),4),  0.0],
})

display(HTML(
    '<div style="font-size:13px">'
    + comparison.style
        .highlight_max(subset=['Test Accuracy','CV Mean Accuracy'], color='#d4edda')
        .highlight_min(subset=['CV Std Dev'], color='#d4edda')
        .format({'Test Accuracy':'{:.4f}','CV Mean Accuracy':'{:.4f}','CV Std Dev':'{:.4f}'})
        .set_table_styles([{'selector':'th','props':[('background','#0f3460'),('color','white'),('font-size','13px')]}])
        .to_html()
    + '</div>'
))

### 7.8 Interactive Burnout Risk Predictor


In [ ]:
# ── Interactive Burnout Risk Predictor ────────────────────────────────────
# Uses the already-trained rf_model from Step 7.3.
# Adjust any slider or dropdown — prediction updates immediately.
# If the widget does not appear, make sure Step 7.3 has been run first.
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
    import numpy as np

    BADGE = {0:('Low Risk','#2ecc71','#fff'), 1:('Medium Risk','#f39c12','#fff'), 2:('High Risk','#e74c3c','#fff')}

    s_genai  = widgets.FloatSlider(value=8.0,  min=0,   max=40,  step=0.5,  description='GenAI hrs/wk:',  style={'description_width':'130px'}, layout=widgets.Layout(width='450px'))
    s_study  = widgets.FloatSlider(value=11.0, min=1,   max=36,  step=0.5,  description='Study hrs/wk:',  style={'description_width':'130px'}, layout=widgets.Layout(width='450px'))
    s_dep    = widgets.IntSlider(  value=3,    min=1,   max=10,  step=1,    description='AI Dependency:',  style={'description_width':'130px'}, layout=widgets.Layout(width='450px'))
    s_anx    = widgets.IntSlider(  value=4,    min=1,   max=10,  step=1,    description='Exam Anxiety:',   style={'description_width':'130px'}, layout=widgets.Layout(width='450px'))
    s_gpa    = widgets.FloatSlider(value=3.2,  min=1.0, max=4.0, step=0.05, description='Pre-Sem GPA:',    style={'description_width':'130px'}, layout=widgets.Layout(width='450px'))
    s_tool   = widgets.IntSlider(  value=2,    min=1,   max=5,   step=1,    description='Tool Diversity:', style={'description_width':'130px'}, layout=widgets.Layout(width='450px'))
    d_skill  = widgets.Dropdown(options=['Beginner','Intermediate','Advanced'], value='Intermediate',
                                description='Prompt Skill:', style={'description_width':'130px'})
    d_major  = widgets.Dropdown(options=['Arts','Business','Humanities','Medical','STEM'], value='STEM',
                                description='Major:', style={'description_width':'130px'})
    d_year   = widgets.Dropdown(options=['Freshman','Sophomore','Junior','Senior','Graduate'], value='Junior',
                                description='Year:', style={'description_width':'130px'})
    d_policy = widgets.Dropdown(options=['Strict_Ban','Allowed_With_Citation','Actively_Encouraged'],
                                value='Allowed_With_Citation', description='Policy:', style={'description_width':'130px'})
    d_use    = widgets.Dropdown(
                                options=['Copywriting/Drafting','Debugging/Troubleshooting',
                                         'Direct_Answer_Generation','Ideation','Summarizing_Reading'],
                                value='Debugging/Troubleshooting', description='AI Use Case:',
                                style={'description_width':'130px'})
    d_paid   = widgets.Checkbox(value=False, description='Paid Subscription')
    out_pred = widgets.Output()

    def predict(change=None):
        with out_pred:
            clear_output(wait=True)
            genai    = s_genai.value;  study  = s_study.value
            dep      = s_dep.value;    anx    = s_anx.value
            gpa      = s_gpa.value;    tool   = s_tool.value
            skill    = {'Beginner':1,'Intermediate':2,'Advanced':3}[d_skill.value]
            year_num = {'Freshman':1,'Sophomore':2,'Junior':3,'Senior':4,'Graduate':5}[d_year.value]
            paid_i   = int(d_paid.value)
            # Derived features (same as Step 5)
            ai_ratio   = round(genai / (study + genai + 1e-6), 4)
            retention  = max(10.0, 100.0 - dep * 4 - genai * 0.5)
            skill_eff  = round(retention / (genai + 1), 4)
            high_dep   = int(dep >= 5)
            post_gpa   = min(4.0, gpa + 0.2)
            gpa_change = round(post_gpa - gpa, 3)
            # Build feature vector
            row = {f: 0.0 for f in FEATURES}
            num_vals = {
                'Pre_Semester_GPA': gpa, 'Post_Semester_GPA': post_gpa,
                'Weekly_GenAI_Hours': genai, 'Tool_Diversity': tool,
                'Paid_Subscription': paid_i, 'Traditional_Study_Hours': study,
                'Perceived_AI_Dependency': dep, 'Anxiety_Level_During_Exams': anx,
                'Skill_Retention_Score': retention, 'Year_of_Study': year_num,
                'Prompt_Engineering_Skill': skill, 'GPA_Change': gpa_change,
                'AI_Study_Ratio': ai_ratio, 'Skill_Efficiency': skill_eff,
                'High_AI_Dependency': high_dep, 'Prompt_Skill_Numeric': skill,
            }
            for k, v in num_vals.items():
                if k in row:
                    row[k] = float(v)
            for feat in FEATURES:
                if feat.startswith('Major_Category_'):
                    row[feat] = 1.0 if d_major.value == feat.replace('Major_Category_','') else 0.0
                elif feat.startswith('Primary_Use_Case_'):
                    row[feat] = 1.0 if d_use.value == feat.replace('Primary_Use_Case_','') else 0.0
                elif feat.startswith('Institutional_Policy_'):
                    row[feat] = 1.0 if d_policy.value == feat.replace('Institutional_Policy_','') else 0.0
            vec = np.array([[row[f] for f in FEATURES]], dtype=float)
            pred  = rf_model.predict(vec)[0]
            proba = rf_model.predict_proba(vec)[0]
            label, bg, fg = BADGE[pred]
            classes = rf_model.classes_.tolist()
            cls_names = {0:'Low', 1:'Medium', 2:'High'}
            bars_html = ''.join(
                '<div style="background:{};width:{}px;height:20px;border-radius:4px;'
                'display:inline-flex;align-items:center;padding:0 6px;margin-right:4px;'
                'color:#fff;font-size:11px;font-weight:700;min-width:44px">{}</div>'.format(
                    '#2ecc71' if c==0 else '#f39c12' if c==1 else '#e74c3c',
                    max(40, int(proba[i]*280)),
                    '{:.0f}%'.format(proba[i]*100)
                ) for i,c in enumerate(classes)
            )
            prob_txt = ' | '.join('{}: {:.1f}%'.format(cls_names[c], proba[i]*100) for i,c in enumerate(classes))
            display(HTML(
                '<div style="font-family:sans-serif;margin-top:10px">'
                '<span style="background:{};color:{};font-size:22px;font-weight:800;'
                'padding:12px 32px;border-radius:8px;letter-spacing:0.5px">Predicted: {}</span>'
                '<div style="margin-top:12px;font-size:13px;color:#555">{}</div>'
                '<div style="margin-top:8px;display:flex;align-items:center;gap:4px">{}</div>'
                '</div>'.format(bg, fg, label, prob_txt, bars_html)
            ))

    for w in [s_genai, s_study, s_dep, s_anx, s_gpa, s_tool, d_skill, d_major, d_year, d_policy, d_use, d_paid]:
        w.observe(predict, names='value')

    from IPython.display import display, HTML
    display(HTML('<b style="font-size:14px;color:#0f3460">&#129302; Burnout Risk Predictor'
                 ' &mdash; adjust any control for an instant prediction</b><br><br>'))
    display(widgets.HBox([
        widgets.VBox([s_genai, s_study, s_dep, s_anx, s_gpa, s_tool],
                     layout=widgets.Layout(border='1px solid #dee2e6', padding='10px',
                                           border_radius='8px', background='#f8f9fa', width='460px')),
        widgets.VBox([d_skill, d_major, d_year, d_policy, d_use, d_paid],
                     layout=widgets.Layout(border='1px solid #dee2e6', padding='10px',
                                           border_radius='8px', background='#f8f9fa', width='320px',
                                           margin_left='10px')),
    ]))
    display(out_pred)
    predict()

except NameError:
    print('rf_model not found. Run Step 7.3 (Random Forest) first, then re-run this cell.')
except ImportError:
    print('ipywidgets not installed. Run Step 0 first, then restart the kernel.')


<div style="display:flex;align-items:center;gap:0;margin:24px 0 8px;font-size:12px;font-weight:700;font-family:sans-serif">
  <div style="background:#0f3460;color:#fff;padding:8px 18px;border-radius:6px 0 0 6px">&#10003; Steps 1&#8211;7 Complete</div>
  <div style="background:#e94560;color:#fff;padding:8px 18px">&#9654; Step 8: Insights</div>
  <div style="background:#dee2e6;color:#555;padding:8px 18px;border-radius:0 6px 6px 0">Step 9: Conclusion</div>
</div>

## Step 8 — Key Analytical Insights

In [ ]:
# ── Compute insights ───────────────────────────────────────────────────────

gpa_by_burnout = (
    df.groupby('Burnout_Risk_Level')['GPA_Change']
      .mean().reindex(['Low','Medium','High']).round(4)
)

corr_dep_skill = df['Perceived_AI_Dependency'].corr(df['Skill_Retention_Score'])
corr_ai_gpa   = df['Weekly_GenAI_Hours'].corr(df['GPA_Change'])

burnout_by_skill = (
    df.groupby('Prompt_Engineering_Skill')['Burnout_Risk_Level']
      .apply(lambda x: (x == 'High').mean() * 100)
      .reindex(['Beginner','Intermediate','Advanced']).round(2)
)

avg_genai_per_burnout = (
    df.groupby('Burnout_Risk_Level')['Weekly_GenAI_Hours']
      .mean().reindex(['Low','Medium','High']).round(2)
)

# ── Styled HTML card ───────────────────────────────────────────────────────
html = '''
<div style="background:#f8f9fa;border-radius:10px;padding:20px 24px;font-family:sans-serif;font-size:13px">
  <h3 style="color:#0f3460;margin-top:0">&#128202; Key Analytical Insights</h3>
  <table style="width:100%;border-collapse:collapse">
    <tr style="background:#0f3460;color:white">
      <th style="padding:8px 12px;text-align:left">#</th>
      <th style="padding:8px 12px;text-align:left">Metric</th>
      <th style="padding:8px 12px;text-align:left">Finding</th>
    </tr>
'''
rows = [
    ('1', 'GPA Change by Burnout Risk',
     f'Low: {gpa_by_burnout["Low"]:+.4f} | Medium: {gpa_by_burnout["Medium"]:+.4f} | High: {gpa_by_burnout["High"]:+.4f}'),
    ('2', 'AI Dependency vs Skill Retention (corr)',
     f'{corr_dep_skill:.4f} — weak negative correlation: higher dependency slightly reduces retention'),
    ('3', 'Weekly GenAI Hours vs GPA Change (corr)',
     f'{corr_ai_gpa:.4f} — weak relationship; other factors dominate GPA changes'),
    ('4', 'High Burnout Rate (%) by Prompt Skill',
     f'Beginner: {burnout_by_skill["Beginner"]}% | Intermediate: {burnout_by_skill["Intermediate"]}% | Advanced: {burnout_by_skill["Advanced"]}%'),
    ('5', 'Avg Weekly GenAI Hours by Burnout Level',
     f'Low: {avg_genai_per_burnout["Low"]}h | Medium: {avg_genai_per_burnout["Medium"]}h | High: {avg_genai_per_burnout["High"]}h'),
    ('6', 'Majority-class Baseline Accuracy', '42.29% — model (Random Forest) achieves ~53%, a genuine +25% relative improvement'),
]
for i, (n, metric, finding) in enumerate(rows):
    bg = '#ffffff' if i % 2 == 0 else '#f1f3f9'
    html += f'<tr style="background:{bg}"><td style="padding:7px 12px">{n}</td><td style="padding:7px 12px;font-weight:600">{metric}</td><td style="padding:7px 12px">{finding}</td></tr>'
html += '</table></div>'
display(HTML(html))

## Step 9 — Conclusion & Ethical Recommendations

<div style="background:linear-gradient(135deg,#f0f4ff,#fff);border:1px solid #dee2e6;border-radius:10px;padding:24px 28px;font-size:13.5px;line-height:1.8">

<h3 style="color:#0f3460;margin-top:0">&#127919; Summary of Findings</h3>

<p>This project analysed <b>50,000 student records</b> to understand how Generative AI tools influence academic outcomes across five major categories and five academic year levels.</p>

<ol>
  <li><b>GPA Impact:</b> Students categorised as <em>High</em> burnout risk show a lower average GPA improvement than <em>Low</em> risk students, confirming that burnout is not merely psychological — it directly suppresses academic performance.</li>
  <li><b>AI Dependency Risk:</b> Higher self-reported AI dependency weakly but consistently correlates with reduced skill retention, suggesting that over-reliance may hollow out genuine learning.</li>
  <li><b>Prompt Engineering as a Protective Factor:</b> Students with <em>Advanced</em> prompt engineering skills have lower high-burnout rates than <em>Beginners</em>, indicating that skill quality mediates AI's impact.</li>
  <li><b>Usage Volume vs Policy:</b> Students under <em>Actively Encouraged</em> institutional policies spend significantly more hours on GenAI tools; this amplifies both the benefits and risks of AI use.</li>
  <li><b>Model Performance:</b> The Random Forest classifier achieved ~53% accuracy vs a 42.3% majority-class baseline — a genuine 25% relative improvement — confirming real but noisy signal in the dataset.</li>
</ol>

<h3 style="color:#0f3460">&#9878;&#65039; Ethical Recommendations</h3>

<table style="width:100%;border-collapse:collapse;font-size:13px">
<tr style="background:#0f3460;color:white">
  <th style="padding:8px 12px">Recommendation</th>
  <th style="padding:8px 12px">Rationale</th>
</tr>
<tr style="background:#f8f9fa">
  <td style="padding:7px 12px"><b>Embed AI Literacy in Curricula</b></td>
  <td style="padding:7px 12px">Prompt engineering should be a formal competency, not an informal skill gap</td>
</tr>
<tr>
  <td style="padding:7px 12px"><b>Adopt Citation-Based AI Policies</b></td>
  <td style="padding:7px 12px">Blanket bans neither reduce anxiety nor improve outcomes; citation-required policies strike the best balance</td>
</tr>
<tr style="background:#f8f9fa">
  <td style="padding:7px 12px"><b>Deploy Early-Warning Systems</b></td>
  <td style="padding:7px 12px">ML burnout predictors can flag at-risk students at mid-semester — with full transparency and student consent</td>
</tr>
<tr>
  <td style="padding:7px 12px"><b>Preserve Traditional Study Time</b></td>
  <td style="padding:7px 12px">Traditional study hours remain among the top positive predictors of GPA improvement; AI must supplement, not replace</td>
</tr>
<tr style="background:#f8f9fa">
  <td style="padding:7px 12px"><b>Data Privacy Governance</b></td>
  <td style="padding:7px 12px">Any deployment of student behavioural data for ML must be governed by explicit consent frameworks and data minimisation principles</td>
</tr>
</table>

<p style="margin-top:20px;color:#666;font-size:12px">
<em>IBM SkillsBuild Data Analytics with AI Academic Internship &mdash; BharatCares &times; AICTE &mdash; Dharamveer Sharma</em>
</p>
</div>